# XMM-Newton 星系 CGM 单源处理流程（组会笔记）

**对象**：EPIC（MOS1 / MOS2 / PN）成像数据  
**目标**：在扣除背景与亮源后，构建干净的表面亮度场与径向轮廓，用于研究星系周介质（CGM）。

> **说明**：下游绘图中的标签与图例使用英文（与脚本一致）；本笔记正文为中文，便于汇报。

---

## 1. 数据与文件约定

### 1.1 输入模板

- 以 **合并 FOV 图像** 为模板：`comb-fovimsky-...fits`（`fovimsky` 表示在天空坐标上重采样/对齐后的 FOV 强度图）。
- 对每个探测器，将文件名前缀 `comb-` 替换为探测器 ID（如 `mos1S001`），得到 **逐探测器** 的 `fovimsky`。

### 1.2 成对文件（与 fovimsky 同名替换关键字）

| 关键字替换 | 物理含义 |
|------------|----------|
| `bkgimsky` | 背景分量（仪器+天空等建模结果） |
| `protimsky` | 点源/紧凑源模型（proprietary timing/sky 语境下常指 **点源贡献**） |
| `expimsky` | 曝光图（有效曝光时间或等效量，用于 net/var 归一化） |
| `maskimsky` | CCD/探测器 **有效区域掩膜**（脚本中约定：**1=保留，0=剔除**） |

### 1.3 Cheese 掩膜

- 文件名形如：`{det}-cheeset.fits`，与对应 `fovimsky` 同目录。
- **Cheese** 通常来自 XMM 管线或后处理，用于标记 **可信曝光区域** / 剔除坏列、芯片间隙等（具体以你本地生成流程为准）。脚本中把 **四舍五入后等于 1** 的像素视为保留。

### 1.4 DS9 区域文件（`.reg`）

- 主 `.reg`：至少包含一个 **ellipse**（星系主体/需排除区域）与一个 **annulus**（背景参考环）。
- 可选 `_out.reg`：**额外需 mask 的区域**（ellipse / circle / box），与主 reg 共用同一像素坐标系。
- `ds9_is_1based=True` 时，对坐标做 **−1** 偏移，以匹配 **FITS/NumPy 0-based** 像素索引。

---

## 2. 总流程概览（按时间顺序）

```mermaid
flowchart LR
  A[Cheese + CCD mask] --> B[减 bkg + protimsky]
  B --> C[曝光阈值]
  C --> D[高斯平滑]
  D --> E[背景 sigma 图]
  E --> F[连通域 + 滞回阈值]
  F --> G[膨胀 + 椭圆/极高值剔除]
  G --> H[逐探测器 final mask]
  H --> I[环内 net/var/exp 求和]
  I --> J[动态径向分箱 + profile]
```

**合并分析**：不对三探测器做“合并 SB 图”展示；仅在 **环带内对 net、var、exp 求和** 后计算径向轮廓与方向分解。

---

## 3. 预处理：Cheese、CCD 与额外区域

### 3.1 最先应用的掩膜

1. **Cheese**：非 1（或非有限）像素 → 在 `fov`、`bkg`、`prot`、`exp` 上置为 `NaN`。
2. **maskimsky（CCD）**：同样要求 **四舍五入为 1** 才保留。

**原理**：在后续任何减背景、平滑、连通域分析之前，先去掉 **无物理意义或不可靠** 的像素，避免它们污染背景统计与源检测。

### 3.2 额外 DS9 mask

- 若存在 `_out.reg`，将其中几何区域转为布尔 mask，对 **SB 与曝光及相关计数** 同步置 `NaN`。

**用途**：手动去掉邻近亮星系、读出边缘、已知 artefact 等脚本难以自动识别的结构。

---

## 4. 净计数、方差与表面亮度

### 4.1 计数与噪声模型（Gaussian 近似）

在 **减背景与点源模型** 之后：

$$
N_{\mathrm{net}} = S - B - P
$$

其中 $S$ 为 FOV 计数率图，`B` 为背景，`P` 为点源（protimsky）贡献。Poisson 计数下，方差常用：

$$
\mathrm{Var}(N_{\mathrm{net}}) \approx S + B + P
$$

（将各分量视作独立 Poisson 过程的近似；若管线已给出更精确误差，可替换此式。）

### 4.2 表面亮度（SB）

脚本将 net 计数率除以曝光 `exp` 与 **像素立体角**（由 `arcsec_per_pixel` 换算为 deg²）：

$$
\mathrm{SB} \propto \frac{N_{\mathrm{net}}}{\mathrm{exp} \cdot \Omega_{\mathrm{pix}}}
$$

误差：

$$
\sigma_{\mathrm{SB}} \propto \frac{\sqrt{\mathrm{Var}}}{\mathrm{exp} \cdot \Omega_{\mathrm{pix}}}}
$$

### 4.3 曝光阈值

- 条件：`exp < exposure_min_valid`（如 20000）→ 整像素链路上置 `NaN`。

**原理**：低曝光处 **信噪比极差**，减背景后的残差会被放大；硬阈值比“分位数裁剪”更透明、可重复。

---

## 5. 平滑与背景参考（MAD sigma）

### 5.1 NaN 感知高斯平滑

对 SB 场做 `gaussian_filter`，但对 **有限像素权重归一化**：缺失像素不参与卷积核分母，避免洞边缘被零填充拉偏。

**作用**：抑制像素噪声、使后续 **连通域 + 阈值** 在空间上更连贯。

### 5.2 背景水平与离散度

在 **annulus 环内**、可选 **排除星系 ellipse**、排除极端高值后，对平滑图取：

- **中位数** `median` 作为背景中心；
- **MAD**（中位数绝对偏差）× 1.4826 作为 **鲁棒 sigma**，近似高斯标准差。

定义 **sigma 图**：

$$
\Sigma = \frac{I_{\mathrm{sm}} - \mu_{\mathrm{bg}}}{\sigma_{\mathrm{bg}}}
$$

**原理**：CGM 环区常含微弱结构；用鲁棒统计可减少尚未 mask 的弱亮斑对背景尺度的牵引。

---

## 6. 弥散“污染”检测：连通域 + 滞回阈值（hysteresis）

### 6.1 内外分区

- 以半径 `r_split_arcmin` 划分 **内区 / 外区**，分别使用 **更严 / 更松** 的种子与生长阈值，兼顾：
  - 内区：星系附近结构复杂，需更强约束抑制弥散亮污染；
  - 外区：背景更均匀，可检测较弱的扩展源残留。

### 6.2 滞回（双阈值）思想

1. **生长掩膜（grow）**：$\Sigma \ge t_{\mathrm{grow}}$ 的像素作为候选连通集合；
2. **种子（seed）**：$\Sigma \ge t_{\mathrm{seed}}$ 且与某连通集合相交 → 该集合保留；否则整集合丢弃。

**效果**：比单阈值更 **抗碎噪声**：噪声像素难以同时满足“高核 + 低缘连通”。

### 6.3 面积滤波与膨胀

- **最小连通面积**：去掉面积过小的斑块（热像素、孤立噪声）。
- **`binary_dilation`**：对正（及可选负）mask 膨胀若干次，**吃掉边缘羽化**，避免减 mask 后环统计仍被晕污染。

### 6.4 负值区域（可选）

`mask_negative_regions=True` 时，对 $\Sigma$ 使用 **对称的负向滞回**（更负为种子），用于去掉大尺度 **过减** 斑块；默认关闭时仅处理正向异常。

> **脚本笔误提示**：若开启负 mask，请确认变量名为 `negative_mask_thresh` / `negative_grow_thresh`，避免出现未定义变量。

### 6.5 最终保留像素（final mask）

在 **finite SB**、**finite 平滑图**、**不在 hot mask**、**可选不在星系椭圆内**、**不在硬上限 `hard_upper_clip` 之上** 等条件同时满足时，像素标记为 **1=保留**。

---

## 7. 多探测器合并：仅在环内对 net / var / exp 求和

### 7.1 求和规则

对每个探测器已应用 **各自 final mask** 的 `net_counts`、`var_counts`、`exp_binned`：

- `net_sum = Σ net_i`（全 NaN 处结果为 NaN）
- `var_sum = Σ var_i`
- `exp_sum = Σ exp_i`

**注意**：这不是简单把三张图“平均成一张合并图”，而是 **在误差传播意义下** 把各探测器的计数与方差在像素上相加（与独立 Poisson 通道近似一致）。

### 7.2 合并 SB（诊断用，不作展示图）

$$
I_{\mathrm{comb}} = \frac{N_{\mathrm{sum}}}{E_{\mathrm{sum}} \cdot \Omega_{\mathrm{pix}}}
$$

用于环内平均背景、负像素计数等诊断；**profile 本身用 net/var/exp 求和** 计算。

### 7.3 背景水平（annulus mean）

在 annulus 内对 `img_combined` 求平均，再换算为 **counts/s/arcmin²**（`sb_conv`），作为图中虚线参考。

---

## 8. 径向轮廓：动态分箱（按累计 S/N）

### 8.1 径向范围

- 仅使用 `rmin_arcmin`–`rmax_arcmin` 之间的像素。
- 半径由 annulus 中心与像素坐标计算，**角分** 由 `arcsec_per_pixel` 换算。

### 8.2 分箱算法（要点）

1. 将有效像素按半径 **排序**；
2. 从内向外累加 `net` 与 `var`；
3. 当累计 **S/N** $\sum N / \sqrt{\sum \mathrm{Var}} \ge \texttt{snr\_threshold}$ 时 **封箱**，并开始新箱；
4. 最后一箱在数据末尾强制闭合；若超出 `rmax` 则截断。

**原理**：远半径处每像素信号弱，用 **自适应更宽的径向 bin** 保证每个 bin 有足够 S/N，轮廓误差条更可比。

### 8.3 每个 bin 的 profile 值

对 bin 内像素：

$$
\bar{I}_k = \frac{\sum_{\in k} N}{\sum_{\in k} E} \cdot \frac{1}{\Omega_{\mathrm{pix}}} \cdot \mathrm{sb\_conv}
$$

$$
\sigma_k = \frac{\sqrt{\sum_{\in k} \mathrm{Var}}}{\sum_{\in k} E} \cdot \frac{1}{\Omega_{\mathrm{pix}}} \cdot \mathrm{sb\_conv}
$$

纵轴单位：**counts/s/arcmin²**；同时给出 **kpc** 横轴（由 `D_Mpc` 换算）。

---

## 9. 方向分解（disk vs perpendicular）

### 9.1 几何定义

- 以星系 ellipse 的 **位置角** `theta_deg` 为参考，将每个像素相对中心的方位角旋转到 **星系主轴坐标系**。
- 将平面分为 **沿盘 ±45°** 与 **垂直盘 ±45°** 两类扇区（脚本中有一处 `mask_disk` / `mask_perp` 的交换，**汇报时请与图例文字逐项核对实际对应关系**）。

### 9.2 与合并 profile 的关系

- 在同一 **动态径向 bin** 下，分别对两扇区内的像素子集做 **net/var/exp 求和**，得到方向分辨的轮廓。
- **底图 scatter**：使用 **第一个探测器** 的 native SB 图作示意（非合并图），仅用于直观展示扇区覆盖。

---

## 10. 阶段性直方图（质量检查）

脚本在多个阶段输出像素值分布对比：

1. RAW SB（cheese+CCD+extra 后、曝光切割前）——可用 **symlog** 展示大动态范围；
2. 曝光切割后；
3. native 网格；
4. 平滑后；
5. 连通域 mask 后；
6. **最终**（含硬上限剔除）。

每条直方图同时对比 **全图** 与 **半径 > `outer_ring_arcmin_for_hist` 的外环** 子样本，用于检查 **外环是否被内区极端值拖尾** 或 **mask 是否过度/不足**。

---

## 11. 主要输出清单（便于组会幻灯片目录）

| 输出 | 内容 |
|------|------|
| `{galaxy}_{det}_smoothed.png` | 平滑图 + annulus / 分割半径 / 星系椭圆 |
| `{galaxy}_{det}_contamination_masks.png` | 原图 + 正（红）/可选负（青）污染轮廓 |
| `{galaxy}_{det}_final_mask.png` | 最终 0/1 掩膜可视化 |
| `{galaxy}_{det}_stage_histograms.png` | 六阶段分布诊断 |
| `{base}_{det}_mask.fits` | 最终 mask（uint8，1=保留） |
| `combined_radial_profile_*.png` | 合并径向轮廓（arcmin / kpc） |
| `combined_direction_profile_*.png` | 方向分解轮廓 |
| `direction_sectors_ref_det.png` | 扇区示意（参考 det1） |

---

## 12. 汇报时可强调的“方法学要点”

1. **掩膜顺序透明**：Cheese → CCD → 可选手动 → 曝光阈值 → 统计驱动 hot mask。
2. **背景尺度鲁棒**：annulus + MAD，减轻未完备源扣除对背景的偏置。
3. **源检测不是单阈值**：滞回 + 连通域面积 + 膨胀，针对 **弥散残留** 与 **点源晕**。
4. **合并策略保守**：不展示合并 SB 图，**profile 在环内对 net/var/exp 物理量求和**，误差与 S/N 分箱一致。
5. **全流程可审计**：阶段直方图 + 每探测器独立 mask FITS，便于复查与复现。

---

### 附：运行代码

将主脚本保存为 `.py` 或在下方单元格 `%run your_script.py` 执行；本笔记本仅承载 **流程与原理** 文档，便于与代码仓库一并版本管理。

In [ ]:
# %matplotlib inline
# %run /path/to/your_xmm_cgm_pipeline.py